# LLM-JP-3 3.7B Instruct — Baseline Generation (Before Fine-Tuning)

Generate stories using the **base model without LoRA** to establish a pre-fine-tuning baseline.
Uses the same cultural prompts as the fine-tuned version for direct comparison.

**Model**: `llm-jp/llm-jp-3-3.7b-instruct`
**Output**: `generated_before_llm_jp.csv` (100 cultural probe stories) + `cultural_showcase_before_llm_jp.csv` (2 long-form stories)
Place this notebook in the `llm-jp-fiction/` folder next to `LLM-JP-FT.ipynb`.

In [1]:
#Suppress warnings
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, message=".*IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Can't initialize amdsmi.*")
warnings.filterwarnings("ignore")

#RDNA3 / ROCm settings (safe to keep even on NVIDIA)
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"
os.environ["HSA_ENABLE_SDMA"] = "0"
os.environ["TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [2]:
#Load HF token from .env if it exists
from pathlib import Path

acctoken = ''

env_path = Path("../.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line.startswith("HF_TOKEN") and "=" in line:
            key, _, value = line.partition("=")
            acctoken = value.strip()
            if acctoken:
                os.environ["HF_TOKEN"] = acctoken
                os.environ["HUGGING_FACE_HUB_TOKEN"] = acctoken

#For Google Colab, uncomment:
#from google.colab import userdata
#acctoken = userdata.get('HF_TOKEN')

print(f"HF token: {'set' if acctoken else 'NOT SET'}")

HF token: set


In [3]:
#Config
MODEL_ID = "llm-jp/llm-jp-3-3.7b-instruct"
OUTPUT_CSV = "generated_before_llm_jp.csv"
SHOWCASE_CSV = "cultural_showcase_before_llm_jp.csv"

#Detect hardware
GPU_TYPE = "cpu"
if torch.cuda.is_available():
    GPU_TYPE = "amd" if torch.version.hip is not None else "nvidia"
    print(f"Hardware: {'AMD GPU (ROCm)' if GPU_TYPE == 'amd' else 'NVIDIA GPU (CUDA)'} detected.")
else:
    print("Hardware: No GPU detected. Defaulting to CPU.")

Hardware: AMD GPU (ROCm) detected.


## Load Base Model (NO LoRA)
This loads the base model without any fine-tuning adapter.

In [4]:
#Load tokenizer
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=acctoken)

#llm-jp-3 pad token setup
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
    print(f"Set pad_token to unk_token: {tokenizer.unk_token!r}")

#Load model in 4-bit (same quantization as fine-tuned version)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading BASE model (no LoRA): {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    token=acctoken,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.eval()
model.config.use_cache = True
print("Base model loaded — NO fine-tuning applied.")

Loading tokenizer: llm-jp/llm-jp-3-3.7b-instruct
Loading BASE model (no LoRA): llm-jp/llm-jp-3-3.7b-instruct


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Base model loaded — NO fine-tuning applied.


## Generation
Same prompts and parameters as the fine-tuned version for fair comparison.

In [5]:
#Story generation helper — identical to fine-tuned version
def generate_story(genre_label, keywords, synopsis, max_new_tokens=300):
    user_msg = (
        f"次のあらすじとジャンルに合う小説の冒頭を書いて。\n"
        f"ジャンル: {genre_label}\n"
        f"キーワード: {keywords}\n"
        f"あらすじ: {synopsis}"
    )
    messages = [{"role": "user", "content": user_msg}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids      = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [6]:
#Cultural comparison prompts — IDENTICAL to fine-tuned notebook
CULTURAL_PROMPTS = [
    # --- Food & Nature ---
    {"id": "pumpkin",          "genre": "ローファンタジー",  "keywords": "秋、収穫祭、農村、食べ物",
     "synopsis": "秋の収穫祭が近づく農村で、主人公が畑でかぼちゃを収穫する場面から物語が始まる。"},
    {"id": "halloween",        "genre": "ローファンタジー",  "keywords": "ハロウィン、秋、仮装、夜",
     "synopsis": "ハロウィンの夜、主人公が飾り付けられた街を歩きながら不思議な出来事に巻き込まれる。"},
    # --- Food culture ---
    {"id": "breakfast",        "genre": "現実世界恋愛",     "keywords": "日常、朝、食事、家族",
     "synopsis": "主人公が家族と一緒に朝ごはんを食べるところから一日が始まる日常の物語。"},
    {"id": "celebration_food", "genre": "現実世界恋愛",     "keywords": "誕生日、パーティー、食べ物、友達",
     "synopsis": "主人公の誕生日パーティーで、友人たちが集まってお祝いの食事を囲む場面。"},
    # --- Family & social structure ---
    {"id": "family_dinner",    "genre": "現実世界恋愛",     "keywords": "家族、夕食、会話、絆",
     "synopsis": "久しぶりに家族全員が集まった夕食の席で、それぞれの近況を語り合う場面。"},
    {"id": "school",           "genre": "ローファンタジー",  "keywords": "学校、教室、友達、青春",
     "synopsis": "新学期の始まり、主人公が新しいクラスメートと出会い友情を育んでいく物語。"},
    # --- Nature & seasons ---
    {"id": "spring",           "genre": "現実世界恋愛",     "keywords": "春、桜、新生活、出会い",
     "synopsis": "春の訪れとともに新生活を始めた主人公が、ある人物との運命的な出会いを果たす。"},
    {"id": "winter",           "genre": "ハイファンタジー",  "keywords": "冬、雪、寒さ、旅",
     "synopsis": "深い雪に覆われた冬の森を旅する主人公が、雪の中に不思議な痕跡を発見する。"},
    # --- Heroism & conflict (narrative structure test — kishotenketsu vs western arc) ---
    {"id": "hero_origin",      "genre": "ハイファンタジー",  "keywords": "勇者、運命、覚醒、旅立ち",
     "synopsis": "平凡な日常を送っていた若者が、ある日突然自分が伝説の勇者であることを告げられる。"},
    {"id": "conflict",         "genre": "ハイファンタジー",  "keywords": "戦い、仲間、犠牲、勝利",
     "synopsis": "長い戦争の末、主人公のパーティーがついに魔王の城へ乗り込む最終決戦の場面。"},
]

#10 stories × 10 prompts = 100 total
STORIES_PER_PROMPT = 10
results = []
total = len(CULTURAL_PROMPTS) * STORIES_PER_PROMPT
count = 0
print(f"--- Generating {total} stories (BASE model, no LoRA) ---")

for p in CULTURAL_PROMPTS:
    for i in range(STORIES_PER_PROMPT):
        story = generate_story(p["genre"], p["keywords"], p["synopsis"])
        results.append({
            "prompt_id":  p["id"],
            "run":        i + 1,
            "genre":      p["genre"],
            "keywords":   p["keywords"],
            "synopsis":   p["synopsis"],
            "story":      story,
            "model":      MODEL_ID,
        })
        count += 1
        if count % 10 == 0:
            print(f"[{count}/{total}] {p['id']} (run {i+1})")

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"--- Saved {len(df)} stories to {OUTPUT_CSV} ---")

--- Generating 100 stories (BASE model, no LoRA) ---
[10/100] pumpkin (run 10)
[20/100] halloween (run 10)
[30/100] breakfast (run 10)
[40/100] celebration_food (run 10)
[50/100] family_dinner (run 10)
[60/100] school (run 10)
[70/100] spring (run 10)
[80/100] winter (run 10)
[90/100] hero_origin (run 10)
[100/100] conflict (run 10)
--- Saved 100 stories to generated_before_llm_jp.csv ---


In [7]:
#Cultural showcase — long-form generation (Western vs Eastern setting)
def generate_long_story(genre_label, keywords, synopsis, max_new_tokens=3500):
    user_msg = (
        f"次のあらすじとジャンルに合う小説を詳しく書いて。できるだけ長く、文化的な描写を豊かに書いて。\n"
        f"ジャンル: {genre_label}\n"
        f"キーワード: {keywords}\n"
        f"あらすじ: {synopsis}"
    )
    messages = [
        {"role": "system", "content": "あなたは日本語の小説家です。文化的な描写を豊かに、できるだけ詳しく書いてください。"},
        {"role": "user", "content": user_msg},
    ]
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    input_ids      = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.85, top_p=0.95, repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


#Story 1 — Western setting
western_story = generate_long_story(
    genre_label="ローファンタジー",
    keywords="ハロウィン、かぼちゃ、アメリカの田舎町、感謝祭、七面鳥、教会、パイ",
    synopsis=(
        "舞台はアメリカの小さな田舎町。ハロウィンの夜、オレンジ色のかぼちゃのランタンが並ぶ通りを、"
        "主人公のエミリーが歩いている。翌週には感謝祭があり、家族全員が集まって七面鳥を囲む予定だ。"
        "しかしその夜、町の古い教会の鐘が突然鳴り始め、不思議な出来事が起き始める。"
    ),
)
print("=== Western Setting (BASE model) ===")
print(western_story[:500])

#Story 2 — Japanese setting
eastern_story = generate_long_story(
    genre_label="ローファンタジー",
    keywords="夏祭り、神社、お盆、和食、かぼちゃの煮物、浴衣、先祖",
    synopsis=(
        "舞台は日本の古い地方の町。お盆の時期、主人公の花は祖母の家を訪ねる。"
        "夕飯にはかぼちゃの煮物や精進料理が並び、家族で先祖の霊を迎える準備をする。"
        "夜になると町の神社で夏祭りが始まり、浴衣姿の人々が集まってくる。"
        "その祭りの夜、花は神社の裏手で見知らぬ少年と出会う。"
    ),
)
print("\n=== Japanese Setting (BASE model) ===")
print(eastern_story[:500])

# Save
showcase_df = pd.DataFrame([
    {"story_id": "western_by_jp_model", "model": MODEL_ID, "setting": "western", "story": western_story},
    {"story_id": "eastern_by_jp_model", "model": MODEL_ID, "setting": "eastern", "story": eastern_story},
])
showcase_df.to_csv(SHOWCASE_CSV, index=False, encoding="utf-8-sig")
print(f"\n--- Saved to {SHOWCASE_CSV} ---")

=== Western Setting (BASE model) ===
物語の舞台となるのは、アメリカ合衆国カリフォルニア州北部にある緑深い山々や静かな湖が広がるのどかな地域である。ここには、毎年10月31日になると奇妙なことが起きるという伝説がある。それは「ジャック・オ・ランタン」と呼ばれる人魂のような形をしたカボチャで作る提灯が、街中を不気味に点けられることから始まる。この地域では、誰もがこの風習を大切に守り続けている。しかし、その年はいつもとは違う気配を感じていた。

エミリーは、そんな風習の中で育った少女で、毎年ハロウィンの日には決まった場所へ行き、友人たちと一緒に過ごしていた。彼女にとって、これが一年中で一番楽しい行事なのだ。だが、今年は違った。いつもなら賑やかだったハロウィンの街も、どこか静まり返っている。エミリーだけでなく、多くの人がこの異変について噂していた。

一方、隣町の長老であるジョンは、感謝祭が近づいてきたことで一層不安になってきていた。彼は亡くなった妻との思い出が詰まった感謝祭に向けて、例年以上に心を込めて準備を進めていたのだ。その裏には、過去に起きた忌まわしい出来事があった。感謝祭とは、感謝を捧げるための特別な日であり、その日に

=== Japanese Setting (BASE model) ===
主人公・花は、長い夏休みのある日を祖父の家で過ごすため、かつての故郷である町に帰省します。そこでは、お盆という特別な行事が行われます。この行事では、亡くなった祖先の霊が戻ってきて、この世の生者たちと交流するという言い伝えがあります。町では毎年8月15日の夕方から16日の早朝にかけて、夏祭りが開催されます。この祭りには、提灯と共に灯された露店が立ち並び、花火が打ち上げられ、盆踊りも盛大に踊られます。地元の子どもたちだけでなく、近隣地域からの客人たちも多く集まり、活気ある賑わいが続きます。

花がお盆の時期によく訪れるのは、祖父母の家があるからで、例年通りその年も、彼女の親類たちが集まるのです。祖父は既に亡くなっているのですが、祖母は健在であり、母と姉と一緒に暮らしています。妹はまだ幼い子供ですが、成長とともに少しずつ少女らしさを見せ始めてきており、花はその変化を見て微笑ましく思っています。また、花自身は中学生になり、友人関係でも悩むこと